# Step 3 — Train ML Model & Evaluate Baselines

With features engineered, this step runs the **training script** and saves versioned artifacts.

## Where the training code lives

The notebook runs the same logic as the CLI:

| What | Location |
|------|----------|
| **Module** | `app.ml.train` → run as `python -m app.ml.train` from the **project root** |
| **Source file** | [`app/ml/train.py`](../app/ml/train.py) — `train()` loads data via `app.ml.dataset`, builds the preprocessing + `RandomForestClassifier` pipeline in `app.ml.features`, splits stratified holdout data, fits the model, compares to a dummy baseline, then writes `joblib` + JSON metadata |

`--no-tune` skips `RandomizedSearchCV` and uses default forest hyperparameters (faster; use full `python -m app.ml.train` for tuning on `roc_auc`).

## What gets produced

- **`artifacts/approval_model.joblib`** — bundle: fitted sklearn `Pipeline` + nested metadata dict
- **`artifacts/approval_model_meta.json`** — reproducibility: paths, row counts, seed, `best_params`, holdout **metrics vs baseline**, sklearn version, optional git SHA

## Process (high level)

1. Stratified train/test split (default 80/20, seed fixed for repeatability).
2. Feature engineering + preprocessor inside the pipeline (same as production training code).
3. Fit random forest (`class_weight="balanced_subsample"` for skewed approval labels).
4. Evaluate on holdout: probabilities → ROC-AUC; hard predictions → accuracy, binary F1; full **`classification_report`** (precision/recall/F1 per class).
5. **Baseline**: `DummyClassifier(strategy="most_frequent")` — “always predict the majority class” (interpret as a no-skill comparator, labeled in logs as blanket-approval style benchmark).

The cells below run the CLI from the repo root, then load `approval_model_meta.json` for a quick structured view of the same numbers printed during training.


### Notebook setup: project root on `sys.path`

The training module is `app.ml.train`, so Python must resolve the top-level **`app`** package. Running notebooks from `notebooks/` usually leaves the cwd one level below the repo root; the next cell sets `PROJECT_ROOT` to the parent of `notebooks/` (or the cwd if you opened the notebook from the repo root) and prepends it to `sys.path` so `import app...` works consistently.

In [1]:
from pathlib import Path
import sys

_CWD = Path.cwd().resolve()
PROJECT_ROOT = _CWD.parent if _CWD.name == "notebooks" else _CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = C:\coding\boltech


### Run training (subprocess)

Spawns the same interpreter as the notebook with `-m app.ml.train`. **`cwd`** is the repo root so `app` resolves the same way as when you run from a terminal in the project root.


In [2]:
import subprocess
import sys

# Delegates to app/ml/train.py → main() → train(..., tune=False when --no-tune).
# cwd=PROJECT_ROOT so imports and relative paths in config match a normal CLI run from repo root.
cmd = [sys.executable, "-m", "app.ml.train", "--no-tune", "--seed", "42"]
print("RUN:", " ".join(cmd))
proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
print(proc.stdout or "(no stdout)")
if proc.stderr:
    print("stderr:", proc.stderr)
# 0 = success; non-zero means the trainer raised or argparse failed — check stderr.
proc.returncode


RUN: c:\coding\boltech\.venv\Scripts\python.exe -m app.ml.train --no-tune --seed 42
--- Model Performance ---
              precision    recall  f1-score   support

           0     0.5000    0.1209    0.1947        91
           1     0.8556    0.9773    0.9124       485

    accuracy                         0.8420       576
   macro avg     0.6778    0.5491    0.5536       576
weighted avg     0.7994    0.8420    0.7990       576

Model Holdout: {'roc_auc': 0.5582530871190665, 'accuracy': 0.8420138888888888, 'f1': 0.9124157844080847}

--- Baseline (No-Skill / Blanket Approve) Performance ---
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        91
           1     0.8420    1.0000    0.9142       485

    accuracy                         0.8420       576
   macro avg     0.4210    0.5000    0.4571       576
weighted avg     0.7090    0.8420    0.7698       576

Baseline Holdout: {'roc_auc': 0.5, 'accuracy': 0.8420138888888888, 'f1':

0

### Reading the training log (`stdout`)

- **`classification_report` (Model)** — Per-class precision, recall, and F1 on the **holdout** slice. For approval problems, compare **recall** on the positive class (catching approvals) vs **precision** (avoiding false approvals); which matters more depends on business cost.
- **`Model Holdout: {...}`** — Single-number summary: **`roc_auc`** (ranking quality of predicted probabilities), **`accuracy`**, **`f1`** (binary F1 on the chosen threshold — sklearn uses 0.5 on `predict`).
- **Baseline block** — Same reports for `DummyClassifier(most_frequent)`: expect weak or zero performance on the minority class. If the forest is only slightly better than baseline, the problem may be too noisy, features too weak, or the split/class balance dominating the metric.
- **`Saved model to ...`** — Confirms `joblib` and JSON metadata were written under your configured artifacts directory (see `app.config` / `.env`).

The next cell reloads those metrics from **`approval_model_meta.json`** so you can inspect them without re-parsing the text log.

In [3]:
import json
from pprint import pprint

from app.config import get_settings

# Same JSON train() wrote next to approval_model.joblib — useful for dashboards or diffing runs.
meta_path = get_settings().artifacts_dir / "approval_model_meta.json"
meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("Top-level keys:", sorted(meta.keys()))
print("\nHoldout metrics (model vs baseline):")
print("  model:   ", meta.get("metrics"))
print("  baseline:", meta.get("baseline_metrics"))
print("\nIf roc_auc is much higher than baseline, the model is learning signal beyond majority-class guessing.")
print("best_params:", meta.get("best_params"))
print("n_rows:", meta.get("n_rows"), "holdout_fraction:", meta.get("holdout_fraction"), "random_state:", meta.get("random_state"))
pprint(meta)


Top-level keys: ['baseline_metrics', 'best_params', 'data_path', 'git_commit', 'holdout_fraction', 'metrics', 'n_rows', 'random_state', 'sklearn_version', 'trained_at']

Holdout metrics (model vs baseline):
  model:    {'roc_auc': 0.5582530871190665, 'accuracy': 0.8420138888888888, 'f1': 0.9124157844080847}
  baseline: {'roc_auc': 0.5, 'accuracy': 0.8420138888888888, 'f1': 0.9142318567389256}

If roc_auc is much higher than baseline, the model is learning signal beyond majority-class guessing.
best_params: {'note': 'tune_off_defaults'}
n_rows: 2880 holdout_fraction: 0.2 random_state: 42
{'baseline_metrics': {'accuracy': 0.8420138888888888,
                      'f1': 0.9142318567389256,
                      'roc_auc': 0.5},
 'best_params': {'note': 'tune_off_defaults'},
 'data_path': 'C:\\coding\\boltech\\claim_use_case_dataset.xlsx',
 'git_commit': None,
 'holdout_fraction': 0.2,
 'metrics': {'accuracy': 0.8420138888888888,
             'f1': 0.9124157844080847,
             'roc_auc

### Optional: train in-process

Same entrypoint the CLI uses: import **`train()`** from [`app/ml/train.py`](../app/ml/train.py) and call it with `tune=False` (equivalent to `--no-tune`) or `tune=True` for randomized search. Use this when you want a debugger inside the training loop without a subprocess.


In [4]:
from app.config import get_settings
from app.ml.train import train

# Returns Path to approval_model.joblib; prints reports to the notebook output like the CLI.
saved = train(tune=False, random_state=42, data_path=get_settings().claim_data_xlsx)
saved


c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


--- Model Performance ---
              precision    recall  f1-score   support

           0     0.5000    0.1209    0.1947        91
           1     0.8556    0.9773    0.9124       485

    accuracy                         0.8420       576
   macro avg     0.6778    0.5491    0.5536       576
weighted avg     0.7994    0.8420    0.7990       576

Model Holdout: {'roc_auc': 0.5582530871190665, 'accuracy': 0.8420138888888888, 'f1': 0.9124157844080847}

--- Baseline (No-Skill / Blanket Approve) Performance ---
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        91
           1     0.8420    1.0000    0.9142       485

    accuracy                         0.8420       576
   macro avg     0.4210    0.5000    0.4571       576
weighted avg     0.7090    0.8420    0.7698       576

Baseline Holdout: {'roc_auc': 0.5, 'accuracy': 0.8420138888888888, 'f1': 0.9142318567389256}
Saved model to C:\coding\boltech\artifacts\approval_model.jobli

c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
c:\coding\boltech\.venv\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


WindowsPath('C:/coding/boltech/artifacts/approval_model.joblib')